In [3]:
import pandas as pd

df = pd.read_csv('week7_clean_data.csv')

# Määra viitekuupäev
today = pd.to_datetime('2025-02-28')

# Veendu, et sale_date on kuupäeva tüüpi
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Arvuta Recency: viimase ostu kuupäev ja päevade arv tänaseni
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

# Arvuta Frequency: iga kliendi ostude arv
frequency = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'frequency']

# Arvuta Monetary: iga kliendi kogukulutus
monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary_value']

# Liida R, F, M ühte tabelisse
rfm = recency.merge(frequency, on='customer_id').merge(monetary, on='customer_id')

# Määra skoorid 1-5 (qcut jagab andmed viieks võrdseks osaks)
# Recency: madal päevade arv = kõrge skoor (5)
rfm['R_score'] = pd.qcut(
    rfm['recency_days'],
    5,
    labels=[5,4,3,2,1]
)

# Frequency ja Monetary: suurem väärtus = kõrge skoor (5)
# Kasutame rank(method='first'), et vältida vigu korduvate väärtuste puhul
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
)

rfm['M_score'] = pd.qcut(
    rfm['monetary_value'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
)

# Muuda skoorid numbriliseks, et neid saaks liita
rfm[['R_score','F_score','M_score']] = rfm[['R_score','F_score','M_score']].astype(int)

# Loo RFM_Score ja segmendid
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

# Prindi segmentide kokkuvõte
segment_summary = rfm['Segment'].value_counts().reset_index()
segment_summary.columns = ['Segment', 'Klientide arv']
segment_summary['Osakaal %'] = round((segment_summary['Klientide arv'] / segment_summary['Klientide arv'].sum()) * 100, 1)

print("RFM SEGMENTIDE KOKKUVÕTE:")
print(segment_summary.to_string(index=False))

RFM SEGMENTIDE KOKKUVÕTE:
      Segment  Klientide arv  Osakaal %
    Potential            759       29.9
        Loyal            679       26.7
      At Risk            531       20.9
VIP Champions            455       17.9
         Lost            116        4.6
